### Libraries



In [1]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from evaluate import load as load_metric
from huggingface_hub import login

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5

### Login to huggingface

In [2]:
login()

### Testing

In [3]:
# === SETTINGS ===
model_id = "eduhuemar001/tinyllama-german"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

Device set to use cuda:0


In [5]:
bleu = load_metric("bleu")
#grammar_prompts = [
#    ("Was ist der Plural von 'Haus'?", "Häuser"),
#    ("Welcher Artikel gehört zu 'Auto'?", "das"),
#    ("Setze den Satz ins Perfekt: Ich gehe zur Schule.", "Ich bin zur Schule gegangen."),
#]

grammar_prompts = [
    ("What is the plural of 'house'?", "houses"),
    ("Which article belongs to 'car'?", "the"),
    ("Put the sentence into present perfect: I go to school.", "I have gone to school."),
]

gen_outputs = []
references = []

print("\n=== German Grammar & Vocabulary ===")
for prompt, expected in grammar_prompts:
    result = generator(prompt, max_new_tokens=30)[0]["generated_text"].replace(prompt, "").strip()
    print(f"Prompt: {prompt}\nModel: {result}\nExpected: {expected}\n")
    gen_outputs.append(result)
    references.append([expected])

print("BLEU Score:", bleu.compute(predictions=gen_outputs, references=references))


=== German Grammar & Vocabulary ===
Prompt: What is the plural of 'house'?
Model: Weblinks
Expected: houses

Prompt: Which article belongs to 'car'?
Model: Siehe auch 

 Liste von Artikelnamen

Einzelnachweise 

Artikel
Artikel
Expected: the

Prompt: Put the sentence into present perfect: I go to school.
Model: In der deutschen Sprache wird die Verwendung des Verbes in der Präsens auch als „Schulpflicht“ bezeichnet.
Expected: I have gone to school.

BLEU Score: {'bleu': 0.0, 'precisions': [0.038461538461538464, 0.0, 0.0, 0.0], 'brevity_penalty': 1.0, 'length_ratio': 3.25, 'translation_length': 26, 'reference_length': 8}


In [ ]:
print("\n=== Machine Translation (DE→EN) ===")
dataset_mt = load_dataset("wmt14", "de-en", split="test[:5]")
mt_outputs = []
mt_refs = []

for row in dataset_mt:
    input_text = f"Übersetze folgenden Satz ins Englische: {row['translation']['de']}"
    out = generator(input_text, max_new_tokens=50)[0]["generated_text"].replace(input_text, "").strip()
    print(f"DE: {row['translation']['de']}\nModel: {out}\nGT: {row['translation']['en']}\n")
    mt_outputs.append(out)
    mt_refs.append([row["translation"]["en"]])

print("BLEU (Translation):", bleu.compute(predictions=mt_outputs, references=mt_refs))


=== Machine Translation (DE→EN) ===


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

DE: Gutach: Noch mehr Sicherheit für Fußgänger
Model: und Autoverkehr.

Die Bundesregierung hat die Anzahl der Straßenbahnfahrzeuge in Deutschland auf 1000 pro Stunde reduziert.

Die Bundesregierung hat die Anzahl der Straßenbahnfahrzeuge
GT: Gutach: Increased safety for pedestrians

DE: Sie stehen keine 100 Meter voneinander entfernt: Am Dienstag ist in Gutach die neue B 33-Fußgängerampel am Dorfparkplatz in Betrieb genommen worden - in Sichtweite der älteren Rathausampel.
Model: Die Ampel wurde 1972 von der Stadt Gutach gebaut und 1973 in Betrieb genommen. Sie ist 100 Meter lang und 1,50 Meter hoch. Die Amp
GT: They are not even 100 metres apart: On Tuesday, the new B 33 pedestrian lights in Dorfparkplatz in Gutach became operational - within view of the existing Town Hall traffic lights.

DE: Zwei Anlagen so nah beieinander: Absicht oder Schildbürgerstreich?
Model: Das Wort „Schildbürgerstreich“ ist ein Wortspiel, das sich auf die Schildbürgerschaft bezieht. Die Schildbürgerschaft i